In [6]:

import pandas as pd

prix = pd.Series([800, 20, 50, 200])

print(prix[3])

200


In [ ]:
import pandas as pd

data = {
    "Produit": ["PC", "Souris", "Clavier", "Écran"],
    "Prix": [800, 20, 50, 200],
    "Quantité": [5, 15, 10, 4],
    "Argent": [1000,995,970,914]
}

df = pd.DataFrame(data)

print(df)

   Produit  Prix  Quantité  Argent
0       PC   800         5    1000
1   Souris    20        15     995
2  Clavier    50        10     970
3    Écran   200         4     914


In [ ]:
import pandas as pd 
sup = pd.read_csv("data/sup_antenne.txt", sep=",")

sup.dtypes

In [ ]:
from pathlib import Path
import pandas as pd

# Repertoire du projet
PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "data"

"""
Si VSCode suprimer : PROJECT_DIR = Path.cwd()
                    DATA_DIR = PROJECT_DIR / "data"

et le remplacer par : PROJECT_DIR = Path(__file__).resolve().parent
                                    DATA_DIR = PROJECT_DIR / "data"
                   
"""

# Fichier data ANFR
files = {
    "antenne": "sup_antenne.txt",
    "bande": "sup_bande.txt",
    "emetteur": "sup_emetteur.txt",
    "station": "sup_station.txt",
    "support": "sup_support.txt",
}

# Chargement des data dans la tables
data = {}

for name, filename in files.items():
  file_path = DATA_DIR / filename

  data[name] = pd.read_csv(file_path, sep=";", low_memory=False)

  print(f"{name}: {len(data[name])} lignes chargées")


# Preparation du support de coordonnées
support = data["support"].copy()

coordinate_columns = [
    "COR_NB_DG_LAT",
    "COR_NB_MN_LAT",
    "COR_NB_SC_LAT",
    "COR_NB_DG_LON",
    "COR_NB_MN_LON",
    "COR_NB_SC_LON",
]

# Conversion des coordonnées en nombres
for column in coordinate_columns:
  support[column] = pd.to_numeric(support[column], errors="coerce")

# Calcul de la latitude
support["latitude"] = (
    support["COR_NB_DG_LAT"]
    + support["COR_NB_MN_LAT"] / 60
    + support["COR_NB_SC_LAT"] / 3600
)

# Calcul de la longitude
support["longitude"] = (
    support["COR_NB_DG_LON"]
    + support["COR_NB_MN_LON"] / 60
    + support["COR_NB_SC_LON"] / 3600
)

# Appliquation géographique
support.loc[
    support["COR_CD_NS_LAT"].astype(str).str.upper() == "S", "latitude"
] *= -1

support.loc[
    support["COR_CD_EW_LON"].astype(str).str.upper() == "W", "longitude"
] *= -1

# Conservation des coordonnées valides
support = support.dropna(subset=["latitude", "longitude"])

# Limiter les coordonnées au territoire métropolitain français
support = support[
    support["latitude"].between(41, 52) & support["longitude"].between(-6, 10)
]

print("Supports valides:", len(support))

In [1]:
from pathlib import Path
import pandas as pd
import folium
from folium.plugins import FastMarkerCluster

# Repertoire du projet
PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "data"

"""
Si VSCode suprimer : PROJECT_DIR = Path.cwd()
                    DATA_DIR = PROJECT_DIR / "data"

et le remplacer par : PROJECT_DIR = Path(__file__).resolve().parent
                                    DATA_DIR = PROJECT_DIR / "data"
                   
"""

# Fichier data ANFR
files = {
    "antenne": "sup_antenne.txt",
    "bande": "sup_bande.txt",
    "emetteur": "sup_emetteur.txt",
    "station": "sup_station.txt",
    "support": "sup_support.txt",
}

# Chargement des data dans la tables
data = {}

for name, filename in files.items():
  file_path = DATA_DIR / filename

  data[name] = pd.read_csv(file_path, sep=";", low_memory=False)

  print(f"{name}: {len(data[name])} lignes chargées")


# Preparation du support de coordonnées
support = data["support"].copy()

coordinate_columns = [
    "COR_NB_DG_LAT",
    "COR_NB_MN_LAT",
    "COR_NB_SC_LAT",
    "COR_NB_DG_LON",
    "COR_NB_MN_LON",
    "COR_NB_SC_LON",
]

# Conversion des coordonnées en nombres
for column in coordinate_columns:
  support[column] = pd.to_numeric(support[column], errors="coerce")

# Calcul de la latitude
support["latitude"] = (
    support["COR_NB_DG_LAT"]
    + support["COR_NB_MN_LAT"] / 60
    + support["COR_NB_SC_LAT"] / 3600
)

# Calcul de la longitude
support["longitude"] = (
    support["COR_NB_DG_LON"]
    + support["COR_NB_MN_LON"] / 60
    + support["COR_NB_SC_LON"] / 3600
)

# Appliquation géographique
support.loc[
    support["COR_CD_NS_LAT"].astype(str).str.upper() == "S", "latitude"
] *= -1

support.loc[
    support["COR_CD_EW_LON"].astype(str).str.upper() == "W", "longitude"
] *= -1

# Conservation des coordonnées valides
support = support.dropna(subset=["latitude", "longitude"])

# Limiter les coordonnées au territoire métropolitain français
support = support[
    support["latitude"].between(41, 52) & support["longitude"].between(-6, 10)
]

print("Supports valides:", len(support))

radio_map = folium.Map(location=[46.6, 2.5], zoom_start=6, tiles=None)

# Fond de carte principal : Esri WorldStreetMap
folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}",
    attr="Tiles &copy; Esri",
    name="Esri WorldStreetMap",
    overlay=False,
    control=True,
    show=True,
).add_to(radio_map)

# Fond alternatif épuré : Esri WorldGrayCanvas
folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/World_Light_Gray_Base/MapServer/tile/{z}/{y}/{x}",
    attr="Tiles &copy; Esri",
    name="Esri Gray Canvas",
    overlay=False,
    control=True,
    show=False,
).add_to(radio_map)

# Prepare geographical coordinates
coordinates = list(zip(support["latitude"], support["longitude"]))

# Add grouped radio installation markers
FastMarkerCluster(
    data=coordinates, name="Installations radioélectriques"
).add_to(radio_map)

# Add layer control
folium.LayerControl().add_to(radio_map)

# Save interactive map
map_path = PROJECT_DIR / "carteradioelectrique.html"

radio_map.save(str(map_path))

print("Map created successfully!")
print("Number of supports displayed:", len(coordinates))
print("Map saved at:", map_path)

ModuleNotFoundError: No module named 'folium'